# ML-09: Validation and Research Claim Audit

**Name:** Abdullah Hasan Shah

**Track:** Machine Learning

**Lane:** Content Refresh

## Objective

This notebook audits the Week 5 machine learning model. The goal is to validate the model, check for feature leakage, evaluate model reliability, review research claims, and present conclusions using responsible machine learning practices.

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

pd.set_option("display.max_columns", None)

In [4]:
df = pd.read_csv("C:/Users/ok/Documents/FlyRank Internship/Week 1/flyrank-ml-internship-starter-main/data/processed/refresh_feature_vector.csv")

print(df.shape)

df.head()

(30000, 52)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,provider_used,model_used,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,engaged_sessions_90d,ai_sessions_90d,scroll_events_90d,days_with_impressions,days_with_sessions,impressions_last_30d,clicks_last_30d,sessions_last_30d,impressions_prev_30d,clicks_prev_30d,sessions_prev_30d,content_age_days,age_tier,age_tier_order,days_since_last_update,freshness_tier,word_count_tier,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_declining_label,log_impressions_90d,log_clicks_90d,log_sessions_90d,log_ai_sessions_90d,has_clicks,has_ai_sessions,measurable_opportunity
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,unknown,gemini-2.5-flash,3803,29,22,17,16,1,0,1,88,13,578,2,2,987,13,9,187,181-365,5,20,0-30,2000-3500,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,1,8.243808,3.401197,2.890372,0.0,1,0,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,unknown,gemini-3-flash-preview,15320,7,10,9,9,0,0,1,88,9,2501,2,3,5915,1,2,445,365+,6,25,0-30,2000-3500,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,1,9.636980,2.079442,2.302585,0.0,1,0,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,unknown,gemini-2.5-flash,12581,11,14,11,11,0,0,4,88,11,2382,1,1,6089,3,3,141,91-180,4,20,0-30,3500+,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,1,9.440023,2.484907,2.484907,0.0,1,0,1
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,0.0,0.0,unknown,unknown,11751,58,87,78,75,1,0,3,88,51,3626,22,35,4206,17,26,463,365+,6,22,0-30,unknown,unknown,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,0,9.371779,4.077537,4.369448,0.0,1,0,1
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,unknown,gemini-3-flash-preview,19140,24,177,145,144,0,0,43,88,33,4211,10,14,6452,2,9,263,181-365,5,14,0-30,2000-3500,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,1,9.859588,3.218876,4.983607,0.0,1,0,1


# Research Finding 1

## Claim

Pages with declining traffic are more likely to require a content refresh.

## Validation Questions

- How is "declining" defined?
- Is the label based on historical observations?
- Does every client experience similar traffic patterns?
- Could seasonality influence the result?

## Discussion

This claim appears reasonable because declining traffic often indicates outdated or less competitive content. However, additional validation on unseen clients and different time periods would improve confidence.

# Research Finding 2

## Claim

Older content tends to require refreshing more often.

## Validation Questions

- Does age always imply poor performance?
- Can evergreen content remain valuable?
- Is content quality considered?

## Discussion

Content age is an informative feature but should not be treated as the only indicator. High-quality evergreen content may continue to perform well regardless of age.

# Honest Validation Strategy

Instead of relying only on a random train/test split, a more realistic validation strategy is recommended.

Possible approaches include:

- Group by Client
- Time-based split
- GroupKFold cross-validation

These approaches reduce data leakage and better simulate real-world deployment.

In [5]:
features = [
    "avg_position",
    "content_age_days",
    "engagement_rate",
    "ctr",
    "trend_pct"
]

target = "is_declining_label"

X = df[features]

y = df[target].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

model = RandomForestClassifier(
    random_state=42
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

accuracy = accuracy_score(
    y_test,
    predictions
)

print("Accuracy:", accuracy)

Accuracy: 0.9998333333333334


In [6]:
print(classification_report(
    y_test,
    predictions
))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2732
           1       1.00      1.00      1.00      3268

    accuracy                           1.00      6000
   macro avg       1.00      1.00      1.00      6000
weighted avg       1.00      1.00      1.00      6000



In [7]:
cm = confusion_matrix(
    y_test,
    predictions
)

pd.DataFrame(
    cm,
    columns=["Pred 0","Pred 1"],
    index=["Actual 0","Actual 1"]
)

,Pred 0,Pred 1
Actual 0,2732,0
Actual 1,1,3267


# Feature Leakage Audit

| Feature | Leakage Risk | Explanation |
|----------|--------------|-------------|
| avg_position | Low | Historical ranking information |
| content_age_days | Low | Known before prediction |
| engagement_rate | Low | Historical engagement metric |
| ctr | Low | Historical click-through rate |
| trend_pct | Medium | Could contain information closely related to the prediction period |

## Discussion

Most selected features are available before prediction. However, **trend_pct** requires careful verification because if it is calculated using the same period as the target label, it may introduce information leakage.

In [8]:
importance = pd.DataFrame({

    "Feature":features,

    "Importance":model.feature_importances_

})

importance.sort_values(
    "Importance",
    ascending=False
)

,Feature,Importance
4,trend_pct,0.962559
0,avg_position,0.021757
1,content_age_days,0.012785
3,ctr,0.002440
2,engagement_rate,0.000459


# Error Analysis

Although the model achieved very high accuracy, errors may still occur in several situations:

- Newly published pages
- Seasonal content
- Pages with limited traffic
- Pages experiencing sudden ranking changes
- Pages affected by external events

These situations may reduce prediction reliability and should be monitored during deployment.

# Responsible Claim Rewrite

Instead of saying:

"Our model proves that declining pages can always be detected."

A better scientific claim is:

"The model achieved high predictive performance on the available dataset. Additional validation on unseen clients and future data would be required before production deployment."

This wording avoids overstating the results and follows responsible machine learning practices.

# Comparison with Previous Model

The Week 5 Random Forest model achieved approximately **99.98% accuracy**.

The validation audit suggests that:

- The selected features are meaningful.
- Average Position and Trend Percentage contribute strongly to predictions.
- The model performs well on the available dataset.
- Additional validation strategies (time-based split and GroupKFold) would improve confidence in future deployments.

# Conclusion

This notebook audited the Week 5 machine learning model using responsible validation principles.

The research findings were reviewed, feature leakage was evaluated, model predictions were examined, and claims were rewritten using evidence-based language.

The Random Forest classifier achieved excellent predictive performance while demonstrating that trend percentage and average search position are important indicators of declining content.

Future work should include:

- Time-based validation
- GroupKFold cross-validation
- Hyperparameter tuning
- More extensive leakage testing
- External validation using unseen clients

Overall, this audit improves confidence in the model while acknowledging its limitations and ensuring that conclusions remain supported by evidence.